# Model Attributes — Architecture Summary for Paper

Generates comprehensive model attributes JSON and summary CSV for all 4 architectures.
Source for Table I (Model Architecture) in the paper.

- Total and trainable parameters
- Layer count (named modules)
- Output classes, input size
- LaTeX-ready table

In [ ]:
OUTPUT_DIR = r'E:\decision_consistency_cifar_outputs'
SEED = 42

In [ ]:
import os, json, csv, random
import numpy as np, torch
from torchvision import models
import warnings; warnings.filterwarnings('ignore')
os.makedirs(os.path.join(OUTPUT_DIR,'tables'),exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device,'| PyTorch:',torch.__version__)

In [ ]:
MODELS={'ResNet-18':models.resnet18(pretrained=True).eval().to(device),'ResNet-50':models.resnet50(pretrained=True).eval().to(device),'VGG-16':models.vgg16(pretrained=True).eval().to(device),'MobileNetV2':models.mobilenet_v2(pretrained=True).eval().to(device)}
print('Models loaded:',list(MODELS.keys()))

In [ ]:
model_info={}
dummy=torch.zeros(1,3,224,224).to(device)
print(f'{"Model":<14} {"Params":>14} {"Trainable":>14} {"Layers":>8} {"Output":>8}'); print('-'*62)
for model_name,model in MODELS.items():
    total_params=sum(p.numel() for p in model.parameters()); trainable_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
    layer_names=[name for name,_ in model.named_modules() if name!='']
    with torch.no_grad(): out=model(dummy); output_classes=out.shape[1]
    model_info[model_name]={'architecture':model_name,'total_parameters':total_params,'trainable_params':trainable_params,'total_layers':len(layer_names),'output_classes':output_classes,'input_size':[1,3,224,224],'pretrained':True,'pretrain_dataset':'ImageNet-1K','device':str(device),'seed':SEED,'pytorch_version':torch.__version__}
    print(f'{model_name:<14} {total_params:>14,} {trainable_params:>14,} {len(layer_names):>8} {output_classes:>8}')
json_path=os.path.join(OUTPUT_DIR,'tables','model_attributes.json')
with open(json_path,'w') as f: json.dump(model_info,f,indent=2)
print('\nSaved:',json_path)

In [ ]:
csv_path=os.path.join(OUTPUT_DIR,'tables','model_summary.csv')
with open(csv_path,'w',newline='') as f:
    writer=csv.writer(f); writer.writerow(['Model','Total_Params','Trainable_Params','Total_Layers','Output_Classes','Pretrain_Dataset','Seed'])
    for mn,info in model_info.items(): writer.writerow([mn,info['total_parameters'],info['trainable_params'],info['total_layers'],info['output_classes'],info['pretrain_dataset'],info['seed']])
print('CSV saved:',csv_path)
print('\n=== Paper Table I ===')
print(f'{"Model":<14} {"Params (M)":>12} {"Layers":>8}  Pretrained'); print('-'*50)
for mn,info in model_info.items(): print(f'{mn:<14} {info["total_parameters"]/1e6:>12.1f} {info["total_layers"]:>8}  {info["pretrain_dataset"]}')

In [ ]:
latex=[r'\begin{table}[htbp]',r'\centering',r'\caption{Architectures Used in DCF Evaluation}',r'\label{tab:arch}',r'\begin{tabular}{lrrrr}',r'\toprule',r'\textbf{Model} & \textbf{Params (M)} & \textbf{Depth} & \textbf{Output} & \textbf{Pretrained} \\',r'\midrule']
for mn,info in model_info.items(): latex.append(f"{mn} & {info['total_parameters']/1e6:.1f} & {info['total_layers']} & {info['output_classes']} & {info['pretrain_dataset']} \\\\")
latex+=[r'\bottomrule',r'\end{tabular}',r'\end{table}']
tex_path=os.path.join(OUTPUT_DIR,'tables','model_summary.tex')
with open(tex_path,'w') as f: f.write('\n'.join(latex))
print('LaTeX saved:',tex_path)